# Continual Learning System — MVP Setup & Verification


---
## 0 — Pre-flight System Check
Must pass before installing anything.

In [1]:
import sys, subprocess, platform

# ── Python version ────────────────────────────────────────────────────────────
py = sys.version_info
assert py >= (3, 10), f"Python 3.10+ required. Got {py.major}.{py.minor}"
print(f"✅  Python {py.major}.{py.minor}.{py.micro}")

# ── OS ───────────────────────────────────────────────────────────────────────
print(f"✅  OS: {platform.system()} {platform.release()}")

# ── CUDA driver check (nvcc or nvidia-smi) ───────────────────────────────────
try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
         "--format=csv,noheader"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    for line in smi.splitlines():
        name, mem, drv, cc = [x.strip() for x in line.split(",")]
        mem_gb = int(mem.replace(" MiB", "")) / 1024
        print(f"✅  GPU detected : {name}")
        print(f"    VRAM        : {mem_gb:.1f} GB  (need ≥54 GB for Qwen3-27B bf16)")
        print(f"    Driver      : {drv}")
        print(f"    Compute cap : {cc}")
        if mem_gb < 54:
            print(f"⚠️  VRAM is below the 54 GB minimum for Qwen3-27B bf16.")
            print("    Switch to Qwen3-27B 4-bit (~14 GB) or Qwen3-35B 4-bit (~18 GB).")
        else:
            headroom = mem_gb - 54
            print(f"    Headroom    : ~{headroom:.0f} GB free after model load — sufficient for LoRA + dual-judge")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("❌  nvidia-smi not found. Confirm NVIDIA drivers are installed and GPU is visible.")
    sys.exit(1)

# ── CUDA toolkit version ──────────────────────────────────────────────────────
try:
    nvcc = subprocess.check_output(["nvcc", "--version"], stderr=subprocess.DEVNULL).decode()
    cuda_ver = [l for l in nvcc.splitlines() if "release" in l][0].split("release ")[1].split(",")[0]
    print(f"✅  CUDA toolkit : {cuda_ver}")
    CUDA_VER = cuda_ver.replace(".", "")
except (subprocess.CalledProcessError, FileNotFoundError, IndexError):
    print("⚠️  nvcc not on PATH. Will infer CUDA version from PyTorch after install.")
    CUDA_VER = "124"   # assume 12.4 for RTX Blackwell; adjust if different

print(f"\nPre-flight passed. Proceeding with CUDA_VER={CUDA_VER}")

✅  Python 3.12.3
✅  OS: Linux 6.8.0-106-generic
✅  GPU detected : NVIDIA RTX PRO 6000 Blackwell Server Edition
    VRAM        : 95.6 GB  (need ≥54 GB for Qwen3-27B bf16)
    Driver      : 580.126.09
    Compute cap : 12.0
    Headroom    : ~42 GB free after model load — sufficient for LoRA + dual-judge
✅  CUDA toolkit : 12.8

Pre-flight passed. Proceeding with CUDA_VER=128


---
## 1 - Install

In [3]:
import sys
import subprocess

def run(cmd):
    subprocess.run(cmd, check=True)

# 1) Ensure uv is available
try:
    run([sys.executable, "-m", "uv", "--version"])
except Exception:
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])

def uv_install(*args):
    run([sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, *args])

# 2) PyTorch + CUDA 12.4
uv_install(
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
)

# 3) Core training / model stack
uv_install(
    "unsloth",
    "transformers",
    "peft",
    "trl",
    "accelerate",
    "bitsandbytes",
    "xformers",
)

# 4) Data / retrieval / utilities
uv_install(
    "datasketch",
    "detoxify",
    "datasets",
    "arxiv",
    "sentence-transformers",
    "safetensors",
    "huggingface_hub",
    "jsonlines",
    "tqdm",
    "numpy",
    "matplotlib",
    "scikit-learn",
)

# 5) Merge / eval / app stack
uv_install(
    "mergekit",
    "evaluate",
    "gradio",
    "fastapi",
    "uvicorn",
)

uv 0.11.13 (x86_64-unknown-linux-gnu)


Using Python 3.12.3 environment at: /workspace/venv
Resolved 27 packages in 262ms
Uninstalled 16 packages in 5.44s
Installed 16 packages in 4.26s
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.2.1.3
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.5.147
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.6.1.9
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.3.1.170
 - nvidia-nccl-cu12==2.27.5
 + nvidia-nccl-cu12==2.21.5
 - nvidia-nvjitlink-cu12==12.8.93
 + nvidia-nvjitlink-cu12==12.4.127
 - nvidia-nvtx-cu12==12.8.90
 + nvidia-nvtx-cu12==12.4.127
 - sympy==1.14.0
 + sympy==1.13.1
 - torch==2.10.0
 + to

---
## 2 — HuggingFace Authentication
Qwen3-27B requires accepting the model license on HuggingFace.  
Log in now so model download in Phase 2 is non-interactive.

In [6]:
from huggingface_hub import get_token, whoami, login

# ── Check if already logged in ────────────────────────────────────────────────
token = get_token()

if token is not None:
    try:
        info = whoami(token=token)
        print(f"✅  Already logged in as: {info['name']}")
    except Exception:
        token = None

if token is None:
    print("No HuggingFace token found.")
    print("Run the cell below after pasting your token from https://huggingface.co/settings/tokens")
    print("Token needs READ access to gated repos (Qwen3-27B requires license acceptance).")
    print()
    print(">>> login(token='hf_YOUR_TOKEN_HERE')")
    print()
    print("Or run from terminal: huggingface-cli login")

# ── Check Qwen3-27B is accessible (metadata only, no download) ────────────────
from huggingface_hub import model_info
MODEL_ID = "Qwen/Qwen3.6-27B"
try:
    info = model_info(MODEL_ID)
    print(f"✅  {MODEL_ID} is accessible on HuggingFace Hub")
    print(f"    License: {info.card_data.get('license', 'check model card')}")
    print(f"    (Model will be downloaded in Phase 2 — not here)")
except Exception as e:
    print(f"⚠️  Cannot access {MODEL_ID}: {e}")
    print("    Ensure you are logged in and have accepted the Qwen3-27B license on the Hub.")

No HuggingFace token found.
Run the cell below after pasting your token from https://huggingface.co/settings/tokens
Token needs READ access to gated repos (Qwen3-27B requires license acceptance).

>>> login(token='hf_YOUR_TOKEN_HERE')

Or run from terminal: huggingface-cli login
✅  Qwen/Qwen3.6-27B is accessible on HuggingFace Hub
    License: apache-2.0
    (Model will be downloaded in Phase 2 — not here)


---
## 6 — Smoke Tests: Every Component That Must Work on Cycle 1
Each test is isolated. A failure here means fix before writing any pipeline code.

In [ ]:
# ── 6A: GPU memory budget ──────────────────────────────────────────────────────
import torch

print("=" * 60)
print("  SMOKE TEST 6A — GPU Memory Budget")
print("=" * 60)

device = torch.device("cuda:0")
props  = torch.cuda.get_device_properties(device)
total_gb = props.total_memory / 1024**3

print(f"  Device : {props.name}")
print(f"  VRAM   : {total_gb:.1f} GB")

# Memory budget assertions from the spec
QWEN3_27B_BF16_GB = 54   # model load footprint
headroom = total_gb - QWEN3_27B_BF16_GB

if total_gb >= 90:
    print(f"  ✅ VRAM is ≥ 90 GB — full headroom ({headroom:.0f} GB) for LoRA + dual-judge")
elif total_gb >= 54:
    print(f"  ⚠️  VRAM is between 54–90 GB (headroom: {headroom:.0f} GB).")
    print("     Dual-judge (vN + anchor simultaneously) may be tight. Use sequential inference.")
else:
    print(f"  ❌ VRAM ({total_gb:.0f} GB) is below 54 GB minimum for Qwen3-27B bf16.")
    print("     Switch to Qwen3-27B 4-bit (~14 GB) and recheck.")

# bf16 allocation test — 1 GB dummy tensor
try:
    dummy = torch.zeros(1024, 1024, 256, dtype=torch.bfloat16, device=device)
    del dummy
    torch.cuda.empty_cache()
    print("  ✅ bf16 allocation test passed (1 GB tensor created and freed)")
except torch.cuda.OutOfMemoryError:
    print("  ❌ OOM on 1 GB bf16 allocation — GPU already has significant memory in use")

In [ ]:
# ── 6B: MinHash Deduplication (Tier 1) ────────────────────────────────────────
from datasketch import MinHash, MinHashLSH

print("=" * 60)
print("  SMOKE TEST 6B — MinHash Dedup (Tier 1 Filter)")
print("=" * 60)

def make_minhash(text: str, num_perm: int = 128) -> MinHash:
    m = MinHash(num_perm=num_perm)
    for word in text.lower().split():
        m.update(word.encode("utf-8"))
    return m

# Simulate a tiny dedup run
docs = [
    "The quick brown fox jumps over the lazy dog",
    "The quick brown fox jumps over the lazy dog",   # exact duplicate
    "Continual learning prevents catastrophic forgetting in neural networks",
    "Continual learning avoids catastrophic forgetting in neural nets",   # near-duplicate
]

lsh = MinHashLSH(threshold=0.8, num_perm=128)
kept, dupes = [], []
for i, doc in enumerate(docs):
    m = make_minhash(doc)
    result = lsh.query(m)
    if result:
        dupes.append(i)
    else:
        lsh.insert(str(i), m)
        kept.append(i)

assert len(kept) == 2 and len(dupes) == 2, f"Expected 2 unique, 2 dupes. Got: kept={kept}, dupes={dupes}"
print(f"  Input  : {len(docs)} documents")
print(f"  Kept   : {len(kept)} (unique)")
print(f"  Duped  : {len(dupes)} (near-duplicates at threshold=0.80)")
print(f"  ✅ MinHash dedup working correctly")

In [ ]:
# ── 6C: Detoxify Toxicity Filter (Tier 1) ─────────────────────────────────────
# First run downloads the detoxify model weights (~250 MB) — expected.
from detoxify import Detoxify

print("=" * 60)
print("  SMOKE TEST 6C — Detoxify (Tier 1 Filter)")
print("=" * 60)
print("  Loading detoxify model (downloads ~250 MB on first run)...")

model = Detoxify("original", device="cuda")

test_texts = [
    "Attention mechanisms improve transformer generalization on long sequences.",
    "I want to hurt everyone and destroy everything.",   # should flag
]

TOXICITY_THRESHOLD = 0.70   # aligned with Tier 1 spec

for text in test_texts:
    scores = model.predict(text)
    tox    = scores["toxicity"]
    flag   = "🚫 REJECTED" if tox > TOXICITY_THRESHOLD else "✅ PASSED"
    print(f"  [{flag}] toxicity={tox:.3f}  |  {text[:60]}")

print(f"\n  ✅ Detoxify running on CUDA — Tier 1 toxicity filter ready")

In [ ]:
# ── 6D: FAISS Anchor Vector Index (Tier 2) ────────────────────────────────────
import faiss
import numpy as np

print("=" * 60)
print("  SMOKE TEST 6D — FAISS Anchor Vector Index (Tier 2)")
print("=" * 60)

# Simulate pre-computed anchor embeddings (50 questions × 768 dims)
DIM        = 768
N_ANCHOR   = 50
np.random.seed(42)
anchor_vecs = np.random.randn(N_ANCHOR, DIM).astype("float32")
faiss.normalize_L2(anchor_vecs)   # cosine similarity via inner product

# Build the index
index = faiss.IndexFlatIP(DIM)   # Inner Product after L2 norm = cosine similarity
index.add(anchor_vecs)

# Query: a candidate document vector
query_vec = np.random.randn(1, DIM).astype("float32")
faiss.normalize_L2(query_vec)
D, I = index.search(query_vec, k=3)   # top-3 nearest anchor questions

print(f"  Index size    : {index.ntotal} anchor embeddings ({DIM}-dim)")
print(f"  Top-3 matches : indices={I[0].tolist()}  cosine_sims={D[0].round(3).tolist()}")

# GPU acceleration check
try:
    res   = faiss.StandardGpuResources()
    gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
    D_gpu, I_gpu = gpu_index.search(query_vec, k=3)
    assert (I_gpu == I).all(), "GPU/CPU results mismatch"
    print("  ✅ FAISS GPU index working — cosine search verified")
except Exception as e:
    print(f"  ⚠️  FAISS GPU unavailable ({e}). CPU index will be used — functional but slower.")
    print("  ✅ FAISS CPU index working")

In [ ]:
# ── 6E: Sentence Transformers — Anchor Embedding Generation ───────────────────
# Used to embed the 50 anchor_benchmark.json Q&A pairs into anchor_embeddings.index
# (Phase 2, Step 08)
from sentence_transformers import SentenceTransformer
import numpy as np

print("=" * 60)
print("  SMOKE TEST 6E — Sentence Transformers (Anchor Embeddings)")
print("=" * 60)
print("  Loading all-MiniLM-L6-v2 (~90 MB, fast, adequate for Tier 2 identity safety)...")

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

sample_anchor_questions = [
    "What is your primary objective?",
    "How do you handle requests that conflict with your values?",
    "Describe your approach to factual uncertainty.",
]

vecs = embedder.encode(sample_anchor_questions, normalize_embeddings=True)
print(f"  Encoded {len(sample_anchor_questions)} questions → shape {vecs.shape}")

# Cosine similarity between q1 and q3 (should be low — different topics)
sim = float(np.dot(vecs[0], vecs[2]))
print(f"  Cosine(q1, q3) = {sim:.3f}  (expected < 0.7 — different topics)")
print("  ✅ Sentence Transformers ready — anchor vector index generation confirmed")

In [ ]:
# ── 6F: Safetensors — adapter_vN.safetensors I/O ─────────────────────────────
import torch
from safetensors.torch import save_file, load_file
import tempfile, os

print("=" * 60)
print("  SMOKE TEST 6F — Safetensors (adapter_vN.safetensors I/O)")
print("=" * 60)

# Simulate a LoRA adapter state dict (rank=16 for 27B means ~200 MB)
# Here we use a tiny representative tensor
adapter_state = {
    "base_model.model.model.layers.16.self_attn.q_proj.lora_A.weight": torch.randn(16, 4096),
    "base_model.model.model.layers.16.self_attn.q_proj.lora_B.weight": torch.randn(4096, 16),
    "base_model.model.model.layers.16.self_attn.v_proj.lora_A.weight": torch.randn(16, 4096),
    "base_model.model.model.layers.16.self_attn.v_proj.lora_B.weight": torch.randn(4096, 16),
}

with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "adapter_v0.safetensors")

    # Save
    save_file(adapter_state, path)
    size_mb = os.path.getsize(path) / 1024**2
    print(f"  Saved adapter_v0.safetensors — {size_mb:.2f} MB (representative 4-layer stub)")

    # Load and verify
    loaded = load_file(path)
    assert set(loaded.keys()) == set(adapter_state.keys()), "Key mismatch after round-trip"
    for k in adapter_state:
        assert torch.allclose(adapter_state[k], loaded[k]), f"Tensor mismatch for {k}"

print("  ✅ Safetensors round-trip (save → load → verify) passed")

In [ ]:
# ── 6G: update_log.jsonl — 15-Metric Schema Validation ───────────────────────
import jsonlines, tempfile, os, json
from datetime import datetime, timezone

print("=" * 60)
print("  SMOKE TEST 6G — update_log.jsonl Schema (15-Metric Instrument Panel)")
print("=" * 60)

# Exact 15 metrics from the spec + required metadata fields
REQUIRED_METRICS = [
    "replay_ratio",
    "instability_score",
    "kl_from_anchor",
    "trust_radius",
    "fast_score_delta",
    "shadow_score_slope_core",
    "shadow_score_slope_rotating",
    "distribution_alarm",
    "routing_switch_rate",
    "buffer_entropy",
    "rollback_stage",
    "consecutive_rollbacks",
    "efficiency_score",
    "mean_delta_norm",
    "interference_score",
]

REQUIRED_META = ["version", "timestamp_utc", "adapter_sha256", "system_state"]
ALL_FIELDS    = REQUIRED_META + REQUIRED_METRICS

# Simulate a Cycle 1 log entry
cycle1_entry = {
    "version":                    "v1",
    "timestamp_utc":              datetime.now(timezone.utc).isoformat(),
    "adapter_sha256":             "a" * 64,   # placeholder — real SHA from hashlib in pipeline
    "system_state":               "ACCEPT",
    # 15 metrics (MVL cycle 1 — placeholders for schema test)
    "replay_ratio":               0.70,
    "instability_score":          0.00,
    "kl_from_anchor":             0.00,
    "trust_radius":               0.10,
    "fast_score_delta":           None,       # None until first benchmark completes
    "shadow_score_slope_core":    None,       # None until cycle 5
    "shadow_score_slope_rotating":None,
    "distribution_alarm":         "NOMINAL",
    "routing_switch_rate":        None,       # None until routing monitor is added (cycle 10)
    "buffer_entropy":             None,
    "rollback_stage":             0,
    "consecutive_rollbacks":      0,
    "efficiency_score":           None,       # added at cycle 30
    "mean_delta_norm":            None,       # added at cycle 5
    "interference_score":         None,
}

# Validate all required fields are present
missing = [f for f in ALL_FIELDS if f not in cycle1_entry]
assert not missing, f"Missing fields in log entry: {missing}"

# Write and read back
with tempfile.NamedTemporaryFile(suffix=".jsonl", delete=False, mode="w") as f:
    tmp_path = f.name

try:
    with jsonlines.open(tmp_path, mode="w") as writer:
        writer.write(cycle1_entry)
    with jsonlines.open(tmp_path) as reader:
        loaded_entry = next(iter(reader))
    assert loaded_entry["version"] == "v1"
    assert len([k for k in loaded_entry if k in REQUIRED_METRICS]) == 15
finally:
    os.unlink(tmp_path)

print(f"  Schema fields  : {len(ALL_FIELDS)} ({len(REQUIRED_META)} meta + {len(REQUIRED_METRICS)} metrics)")
print(f"  Round-trip     : write → read → validate passed")
print(f"  None-safe      : metrics that aren't active until later cycles log as null — correct")
print(f"  ✅ update_log.jsonl schema validated")

print("\n  15-metric instrument panel fields:")
for i, m in enumerate(REQUIRED_METRICS, 1):
    print(f"  {i:>2}. {m}")

In [ ]:
# ── 6H: LoRA + PEFT — Adapter Attachment Smoke Test ──────────────────────────
# Uses a tiny GPT-2 as a stand-in for Qwen3-27B to verify LoRA attaches and trains.
# Qwen3-27B is NOT downloaded here.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

print("=" * 60)
print("  SMOKE TEST 6H — LoRA Adapter (PEFT, stand-in model)")
print("=" * 60)
print("  Loading GPT-2 (small stand-in) to verify LoRA config ...")
print("  (Qwen3-27B is downloaded in Phase 2, Step 06 — not here)")

model_id  = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16).cuda()

total_params = sum(p.numel() for p in base_model.parameters())

# LoRA config mirroring the spec: rank=16, alpha=32
# Zone 2 layers (16–25 for 27B) → for GPT-2 we target all attention projections
lora_config = LoraConfig(
    task_type    = TaskType.CAUSAL_LM,
    r            = 16,
    lora_alpha   = 32,
    lora_dropout = 0.05,
    target_modules = ["c_attn"],   # GPT-2 equivalent of q_proj/v_proj in Qwen3
    bias         = "none",
)

peft_model = get_peft_model(base_model, lora_config)
trainable  = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
ratio      = trainable / total_params

print(f"  Total params    : {total_params:,}")
print(f"  Trainable params: {trainable:,}  ({ratio*100:.3f}% of total)")

# INVARIANT CHECK from spec: trainable_params / total_params < 0.02
assert ratio < 0.02, f"INVARIANT VIOLATED: {ratio:.4f} ≥ 0.02. Tighten LoRA rank or target_modules."
print(f"  Invariant check : trainable < 2% — ✅ PASSED ({ratio*100:.3f}%)")

# Quick forward pass to confirm bf16 LoRA trains on GPU
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer("Continual learning in LLMs", return_tensors="pt").to("cuda")
outputs = peft_model(**inputs, labels=inputs["input_ids"])
outputs.loss.backward()

print(f"  Forward+backward pass: loss={outputs.loss.item():.4f} — no errors")
print(f"  ✅ LoRA adapter (rank=16, alpha=32) attaches and trains correctly")

# Cleanup
del peft_model, base_model
torch.cuda.empty_cache()

In [ ]:
# ── 6I: DARE-TIES Merge — Config Validation ────────────────────────────────────
# Verifies merge_spec.yaml config is parseable and mergekit can be invoked.
# An actual merge requires vN + adapter — not run here.
import yaml, tempfile, os, subprocess, sys

print("=" * 60)
print("  SMOKE TEST 6I — DARE-TIES Merge Config")
print("=" * 60)

# merge_spec.yaml as defined in the spec (seed=42, density=0.70)
MERGE_SPEC = {
    "merge_method": "dare_ties",
    "base_model": {"model": "Qwen/Qwen3-27B"},
    "models": [
        {
            "model": "./adapter_vN",
            "parameters": {
                "density": 0.70,
                "weight":  1.0,
            }
        }
    ],
    "parameters": {
        "int8_mask":     True,
        "random_seed":   42,
    },
    "dtype": "bfloat16",
    "tokenizer_source": "base",
}

with tempfile.NamedTemporaryFile(
    suffix=".yaml", delete=False, mode="w"
) as f:
    yaml.dump(MERGE_SPEC, f, default_flow_style=False)
    tmp_yaml = f.name

try:
    # Verify the YAML round-trips correctly
    with open(tmp_yaml) as f:
        loaded = yaml.safe_load(f)

    assert loaded["merge_method"] == "dare_ties"
    assert loaded["parameters"]["random_seed"] == 42
    assert loaded["models"][0]["parameters"]["density"] == 0.70
    assert loaded["dtype"] == "bfloat16"

    print("  merge_spec.yaml schema:")
    print(f"    merge_method : {loaded['merge_method']}")
    print(f"    density      : {loaded['models'][0]['parameters']['density']}")
    print(f"    random_seed  : {loaded['parameters']['random_seed']}")
    print(f"    dtype        : {loaded['dtype']}")

    # Verify mergekit CLI is on PATH
    result = subprocess.run(
        [sys.executable, "-m", "mergekit.scripts.merge", "--help"],
        capture_output=True, text=True
    )
    if result.returncode == 0 or "usage" in (result.stdout + result.stderr).lower():
        print("  mergekit CLI   : accessible via python -m mergekit.scripts.merge")
    else:
        # Try the direct CLI entry point
        result2 = subprocess.run(
            ["mergekit-merge", "--help"],
            capture_output=True, text=True
        )
        if result2.returncode == 0:
            print("  mergekit CLI   : accessible via mergekit-merge")
        else:
            print("  ⚠️  mergekit CLI not found on PATH. Will invoke programmatically in pipeline.")

    print("  ✅ DARE-TIES config validated — ready for Phase 2, Step 11")
finally:
    os.unlink(tmp_yaml)

In [ ]:
# ── 6J: arXiv API — Phase 2 Data Source ──────────────────────────────────────
import arxiv

print("=" * 60)
print("  SMOKE TEST 6J — arXiv API (Phase 2 data source)")
print("=" * 60)

client = arxiv.Client()
search = arxiv.Search(
    query      = "continual learning catastrophic forgetting",
    max_results = 3,
    sort_by    = arxiv.SortCriterion.SubmittedDate,
)

results = list(client.results(search))
assert len(results) > 0, "arXiv returned no results — check network connectivity"

print(f"  Query          : continual learning catastrophic forgetting")
print(f"  Results        : {len(results)} papers fetched")
for r in results:
    print(f"    [{r.published.year}] {r.title[:70]}")

print("  ✅ arXiv API reachable — data source for Phase 2, Step 07 confirmed")

---
## 7 — Final Setup Checklist
All smoke tests aggregated into a single pass/fail summary.  
Every item must show ✅ before you write a single line of pipeline code.

In [7]:
import torch, importlib, subprocess, sys

print("=" * 70)
print("  CONTINUAL LEARNING MVP — FINAL SETUP CHECKLIST")
print("=" * 70)

checks = []

def check(label: str, fn):
    try:
        result = fn()
        status = "✅" if result else "❌"
        detail = "" if result else "  ← FAILED"
    except Exception as e:
        status = "❌"
        detail = f"  ← {type(e).__name__}: {e}"
        result = False
    checks.append(result)
    print(f"  {status}  {label}{detail}")

# ── System ────────────────────────────────────────────────────────────────────
print("\n  [ SYSTEM ]")
check("Python ≥ 3.10",
      lambda: sys.version_info >= (3, 10))
check("CUDA GPU visible",
      lambda: torch.cuda.is_available())
check("VRAM ≥ 54 GB (Qwen3-27B bf16 minimum)",
      lambda: torch.cuda.get_device_properties(0).total_memory / 1024**3 >= 54)
check("bf16 tensors on CUDA",
      lambda: bool(torch.tensor([1.0], dtype=torch.bfloat16).cuda()))

# ── Core ML Stack ─────────────────────────────────────────────────────────────
print("\n  [ CORE ML STACK ]")
for pkg in ["torch", "transformers", "peft", "trl", "accelerate", "bitsandbytes", "unsloth"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── Data Pipeline ─────────────────────────────────────────────────────────────
print("\n  [ DATA PIPELINE ]")
for pkg, imp in [
    ("datasketch (MinHash Tier 1)",        "datasketch"),
    ("detoxify (Toxicity Tier 1)",         "detoxify"),
    ("datasets (HuggingFace)",             "datasets"),
    ("arxiv (Phase 2 data source)",        "arxiv"),
    ("faiss (Anchor vector index Tier 2)", "faiss"),
    ("sentence_transformers (Embeddings)", "sentence_transformers"),
]:
    check(pkg, lambda i=imp: bool(importlib.import_module(i)))

# ── Merge & Versioning ────────────────────────────────────────────────────────
print("\n  [ MERGE & VERSIONING ]")
for pkg in ["safetensors", "mergekit", "evaluate"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── Logging & Utilities ───────────────────────────────────────────────────────
print("\n  [ LOGGING & UTILITIES ]")
for pkg in ["jsonlines", "yaml", "numpy", "matplotlib", "sklearn", "tqdm"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── API & Demo Layer ──────────────────────────────────────────────────────────
print("\n  [ API & DEMO LAYER ]")
for pkg in ["gradio", "fastapi", "uvicorn"]:
    check(f"{pkg} importable",
          lambda p=pkg: bool(importlib.import_module(p)))

# ── HuggingFace Auth ──────────────────────────────────────────────────────────
print("\n  [ HUGGINGFACE AUTH ]")
from huggingface_hub import HfFolder
check("HF token present",
      lambda: HfFolder.get_token() is not None)

# ── Summary ───────────────────────────────────────────────────────────────────
passed = sum(checks)
total  = len(checks)
failed = total - passed

print()
print("=" * 70)
if failed == 0:
    print(f"  🟢  ALL {total} CHECKS PASSED — environment is ready for Phase 1, Day 1")
    print()
    print("  Next step: Phase 1, Step 01")
    print("  → Load Qwen3-27B bf16, verify 42 GB headroom, run inference test.")
else:
    print(f"  🔴  {failed} / {total} CHECKS FAILED")
    print("  Resolve all failures before proceeding.")
    print("  Re-run only the failed cells, then re-run this checklist cell.")
print("=" * 70)

  CONTINUAL LEARNING MVP — FINAL SETUP CHECKLIST

  [ SYSTEM ]
  ✅  Python ≥ 3.10
  ✅  CUDA GPU visible
  ✅  VRAM ≥ 54 GB (Qwen3-27B bf16 minimum)
  ✅  bf16 tensors on CUDA

  [ CORE ML STACK ]
  ✅  torch importable
  ✅  transformers importable


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/workspace/venv/lib/python3.12/site-packages/torchao/quantization/quant_api.py:1731: SyntaxWarning: invalid escape sequence '\.'
  """Configuration class for applying different quantization configs to modules or parameters based on their fully qualified names (FQNs).


  ✅  peft importable
  ✅  trl importable
  ✅  accelerate importable
  ✅  bitsandbytes importable


/usr/lib/python3.12/importlib/__init__.py:90: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  return _bootstrap._gcd_import(name[level:], package, level)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
  ✅  unsloth importable

  [ DATA PIPELINE ]
  ✅  datasketch (MinHash Tier 1)
  ✅  detoxify (Toxicity Tier 1)
  ✅  datasets (HuggingFace)
  ✅  arxiv (Phase 2 data source)
  ❌  faiss (Anchor vector index Tier 2)  ← ModuleNotFoundError: No module named 'faiss'
  ✅  sentence_transformers (Embeddings)

  [ MERGE & VERSIONING ]
  ✅  safetensors importable
  ✅  mergekit importable
  ✅  evaluate importable

  [ LOGGING & UTILITIES ]
  ✅  jsonlines importable
  ✅  yaml importable
  ✅  numpy importable
  ✅  matplotlib importable
  ✅  sklearn importable
  ✅  tqdm importable

  [ API & DEMO LAYER ]
  ✅  gradio importable
  ✅  fastapi importable
  ✅  uvicorn importable

  [ HUGGINGFACE AUTH ]


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (/workspace/venv/lib/python3.12/site-packages/huggingface_hub/__init__.py)

---
# Base Model Loading & v0 Baseline Generation

In [10]:
# ==============================================================================
# BLOCK 01 — Base Model Initialization & v0 Baseline Generation
# ==============================================================================
import json
import torch
import os
from tqdm import tqdm
from unsloth import FastLanguageModel

# Configuration
MODEL_ID = "Qwen/Qwen3.6-27B" # Change if the HF repo name differs slightly
MAX_SEQ_LENGTH = 2048
BENCHMARK_PATH = "anchor_benchmark.json"
BASELINE_OUTPUT_PATH = "anchor_benchmark_v0_answers.json"

print("=" * 60)
print("  BLOCK 01: LOADING BASE MODEL AND GENERATING v0 BASELINE")
print("=" * 60)

# 1. Verify JSON exists
assert os.path.exists(BENCHMARK_PATH), f"❌ {BENCHMARK_PATH} not found. Please generate it using the prompts."
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

assert len(benchmark_data) == 50, f"❌ Expected 50 questions, found {len(benchmark_data)}"

# 2. Load Model using Unsloth (for peak memory efficiency and speed)
print(f"Loading {MODEL_ID} in bfloat16...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16, # Perfect for your Blackwell GPU
    load_in_4bit=False,   # No need for 4-bit, we have 96GB VRAM
    # token="YOUR_HF_TOKEN" # Uncomment and add if model is gated
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
print("✅ Model and Tokenizer loaded successfully.")

# 3. Generate v0 Baseline Answers
print("Generating v0 baseline answers for 50 anchor questions...")
v0_answers =[]

for item in tqdm(benchmark_data, desc="Evaluating Anchor Benchmark"):
    # Using Qwen's standard chat template
    messages =[
        {"role": "system", "content": "You are a helpful, harmless, and honest AI assistant."},
        {"role": "user", "content": item["question"]}
    ]
    
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")
    
    # Generate with anti-repetition settings for stable baseline
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        do_sample=False,              # Keep it deterministic
        repetition_penalty=1.15,      # 🐛 FIXED: Prevents the infinite repeating loop
        pad_token_id=tokenizer.eos_token_id # 🐛 FIXED: Ensures proper stopping
    )
    
    # Decode only the newly generated tokens
    input_length = inputs["input_ids"].shape[1]
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    
    v0_answers.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "expected_keywords": item.get("expected_keywords", []),
        "v0_response": response
    })

# 4. Save the Baseline
print(f"Saving baseline to {BASELINE_OUTPUT_PATH}...")
with open(BASELINE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(v0_answers, f, indent=2)

print(f"✅ Phase 1, Step 01 Complete. {BASELINE_OUTPUT_PATH} created.")
print(f"Next: Tier 1 Data Ingestion (Block 02).")

  BLOCK 01: LOADING BASE MODEL AND GENERATING v0 BASELINE
Loading Qwen/Qwen3.6-27B in bfloat16...
==((====))==  Unsloth 2026.5.2: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights: 100%|██████████| 1184/1184 [01:31<00:00, 12.99it/s] 


✅ Model and Tokenizer loaded successfully.
Generating v0 baseline answers for 50 anchor questions...


Evaluating Anchor Benchmark: 100%|██████████| 50/50 [11:27<00:00, 13.76s/it]

Saving baseline to anchor_benchmark_v0_answers.json...
✅ Phase 1, Step 01 Complete. anchor_benchmark_v0_answers.json created.
Next: Tier 1 Data Ingestion (Block 02).


---
###💻 Block 02: Tier 1 Filter (Statistical)

In [ ]:
# ==============================================================================
# BLOCK 02 — Tier 1 Filter (MinHash + Detoxify)
# ==============================================================================
from datasketch import MinHash, MinHashLSH
from detoxify import Detoxify
import torch

def run_tier1_filter(raw_texts, threshold=0.85, toxicity_limit=0.05):
    print(f"Starting Tier 1 Filter: {len(raw_texts)} samples...")
    
    # 1. Deduplication
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    unique_texts = []
    
    for i, text in enumerate(raw_texts):
        m = MinHash(num_perm=128)
        for word in text.lower().split():
            m.update(word.encode('utf-8'))
        
        # If no match found in LSH, it's unique
        if not lsh.query(m):
            lsh.insert(f"idx_{i}", m)
            unique_texts.append(text)
            
    print(f"  ↳ Deduplication: Kept {len(unique_texts)} / {len(raw_texts)}")
    
    # 2. Toxicity Check
    print("  ↳ Running Toxicity Filter (GPU)...")
    tox_model = Detoxify('original', device='cuda')
    results = tox_model.predict(unique_texts)
    
    clean_texts = [
        txt for i, txt in enumerate(unique_texts) 
        if results['toxicity'][i] < toxicity_limit
    ]
    
    print(f"  ↳ Toxicity: Kept {len(clean_texts)} / {len(unique_texts)}")
    return clean_texts

raw_data = json.load(open("raw_ai_data.json"))
clean_data = run_tier1_filter(raw_data)

---
💻 Block 03: Tier 2 MVL Judge (Model-in-the-loop)

In [ ]:
# ==============================================================================
# BLOCK 03A (MVL COMPLIANT) — Tier 2 Judge
# ==============================================================================
import re
import os
import json
from tqdm import tqdm

print("=" * 60)
print("  BLOCK 03A: TIER 2 JUDGE & FLAG GENERATION")
print("=" * 60)

FastLanguageModel.for_inference(model)

auto_approved = []
flagged_for_review =[]

for text in tqdm(clean_data, desc="Tier 2 Scoring"):
    prompt = f"""Rate the following text for factual quality and relevance to AI research. 
Score: 1 (Poor) to 5 (Excellent). Return ONLY the integer score.

Text: {text}
Score:"""
    
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=2, temperature=0.0)
    score_str = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    match = re.search(r'[1-5]', score_str)
    if match:
        score = int(match.group())
        # MVL Logic: 5 is auto-approve. 3 and 4 are flagged. 1 and 2 are rejected.
        if score == 5:
            auto_approved.append(text)
        elif score in [3, 4]:
            flagged_for_review.append(text)

print(f"\n✅ Tier 2 Complete.")
print(f"   ↳ Auto-approved (Score 5): {len(auto_approved)}")
print(f"   ↳ Flagged for review (Score 3-4): {len(flagged_for_review)}")
print(f"   ↳ Auto-rejected (Score 1-2): {len(clean_data) - len(auto_approved) - len(flagged_for_review)}")

# Save auto-approved state so the next cell can access it
with open("tier2_auto_approved.json", "w", encoding="utf-8") as f:
    json.dump(auto_approved, f)

# Write flagged documents to a simple text file
REVIEW_FILE = "tier3_manual_review.txt"
with open(REVIEW_FILE, "w", encoding="utf-8") as f:
    for text in flagged_for_review:
        f.write(text.replace('\n', ' ') + "\n")

print(f"\n⚠️  ACTION REQUIRED BEFORE RUNNING NEXT CELL:")
print(f"   1. Open '{REVIEW_FILE}' from your Jupyter file browser.")
print(f"   2. Delete the entire line for any document you wish to REJECT.")
print(f"   3. Save the file, then run Block 03B.")

In [ ]:
# ==============================================================================
# BLOCK 03B (MVL COMPLIANT) — Tier 3 Human Review Ingestion
# ==============================================================================

import os
import json

print("=" * 60)
print("  BLOCK 03B: TIER 3 HUMAN REVIEW INGESTION")
print("=" * 60)

AUTO_APPROVED_PATH = "tier2_auto_approved.json"
REVIEW_FILE = "tier3_manual_review.txt"

# ----------------------------------------------------------------------
# 1. Load Tier-2 Auto Approved Data
# ----------------------------------------------------------------------
assert os.path.exists(AUTO_APPROVED_PATH), \
    f"❌ {AUTO_APPROVED_PATH} missing. Run Block 03 first."

with open(AUTO_APPROVED_PATH, "r", encoding="utf-8") as f:
    auto_approved = json.load(f)

print(f"Loaded Tier-2 Auto-Approved Samples: {len(auto_approved)}")

# ----------------------------------------------------------------------
# 2. Load Tier-3 Human Approved Data
# ----------------------------------------------------------------------
human_approved = []

if os.path.exists(REVIEW_FILE):
    with open(REVIEW_FILE, "r", encoding="utf-8") as f:
        for line in f:
            cleaned_line = line.strip()

            # Non-empty line = human kept it
            if cleaned_line:
                human_approved.append(cleaned_line)

    print(f"✅ Tier 3 Complete.")
    print(f"   ↳ Human approved: {len(human_approved)} samples")
else:
    print(f"⚠️  {REVIEW_FILE} not found.")
    print("   ↳ Assuming no human-approved samples.")

# ----------------------------------------------------------------------
# 3. Combine Final Approved Dataset
# ----------------------------------------------------------------------
approved_data = auto_approved + human_approved

print("\n🚀 Final Approved Data for Training")
print(f"   ↳ Auto-approved: {len(auto_approved)}")
print(f"   ↳ Human-approved: {len(human_approved)}")
print(f"   ↳ TOTAL: {len(approved_data)} samples")

print("\n➡️  Next Step: Proceed to Block 04 (Replay Buffer Mixing)")

In [ ]:
# ==============================================================================
# BLOCK 04 (MVL COMPLIANT) — Flat Replay Buffer & Batching
# ==============================================================================
import os
import json
import random
import sys

print("=" * 60)
print("  BLOCK 04: REPLAY BUFFER MIXING (FLAT MVL VERSION)")
print("=" * 60)

REPLAY_PATH = "replay_buffer_seed.json"

assert os.path.exists(REPLAY_PATH), f"❌ {REPLAY_PATH} missing."

# 1. Load Flat Replay Buffer (No Anchor logic, No Stratification)
with open(REPLAY_PATH, "r", encoding="utf-8") as f:
    flat_replay_buffer = json.load(f)

print(f"Loaded Flat Replay Buffer: {len(flat_replay_buffer)} total samples")

new_data_count = len(approved_data) # From Block 03 Tier 2 Judge
if new_data_count == 0:
    print("❌ No new data passed the Tier 2 Judge. Halting Cycle 1.")
    sys.exit(1)

# 2. Hardcoded MVL Ratio: 30% New Data, 70% Replay
replay_count = int((new_data_count / 0.30) * 0.70)

# Randomly sample from the flat buffer (using choices to allow replacement if buffer is small)
sampled_replay = random.choices(flat_replay_buffer, k=replay_count)

# 3. Combine and Shuffle
training_batch_texts = approved_data + sampled_replay
random.shuffle(training_batch_texts)

print(f"✅ Training Batch Ready: {len(training_batch_texts)} total samples.")
print(f"   ↳ {new_data_count} New Samples (Tier 2 Approved)")
print(f"   ↳ {replay_count} Replay Samples (Randomly drawn from flat buffer)")

In [ ]:
# ==============================================================================
# BLOCK 05 — LoRA Training Loop (Unsloth)
# ==============================================================================
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print("=" * 60)
print("  BLOCK 05: LORA TRAINING (Zone 2: Layers 8-24)")
print("=" * 60)

# 1. Format Data into ChatML structure for SFT
def format_prompts(examples):
    texts = []
    for text in examples["text"]:
        # Wrapping the text in standard instruction format so it learns to reply usefully
        prompt = tokenizer.apply_chat_template([
                {"role": "system", "content": "You are a helpful, harmless, and honest AI assistant."},
                {"role": "user", "content": "Explain a useful fact, concept, or snippet of knowledge."},
                {"role": "assistant", "content": text}
            ],
            tokenize=False
        )
        texts.append(prompt)
    return {"formatted_text": texts}

dataset = Dataset.from_dict({"text": training_batch_texts})
dataset = dataset.map(format_prompts, batched=True)

# 2. Attach LoRA Adapters (Strict MVL Rules: r=16, alpha=32, target Zone 2)
# We pass layers_to_transform to PEFT to ensure only layers 8-24 are touched.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    lora_dropout = 0,
    target_modules =["q_proj", "k_proj", "v_proj", "o_proj"],
    layers_to_transform = list(range(8, 25)), # Zone 2 isolation
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

# Verify trainable parameters invariant (< 2%)
trainable_params, total_params = model.get_nb_trainable_parameters()
trainable_pct = (trainable_params / total_params) * 100
print(f"Trainable Parameters: {trainable_params:,} / {total_params:,} ({trainable_pct:.3f}%)")
assert trainable_pct < 2.0, "❌ Invariant Failed: Trainable parameters exceed 2.0%"

# 3. Configure Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False, # Keep false for small MVP datasets to ensure exact step counts
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        learning_rate = 1e-4,
        fp16 = False,
        bf16 = True,       # RTX Blackwell native bfloat16
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        num_train_epochs = 2, # MVL runs 1-3 epochs
        output_dir = "outputs",
    ),
)

# 4. Train!
print("🚀 Starting LoRA Training...")
trainer_stats = trainer.train()

# 5. Save the Adapter (v1)
ADAPTER_SAVE_PATH = "adapter_v1"
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"✅ Training Complete. Adapter saved to ./{ADAPTER_SAVE_PATH}")

In [ ]:
# ==============================================================================
# BLOCK 06 (MVL COMPLIANT) — Evaluation & Binary Rollback Decision
# ==============================================================================
import json

print("=" * 60)
print("  BLOCK 06: EVALUATION & BINARY ROLLBACK DECISION")
print("=" * 60)

FastLanguageModel.for_inference(model)

# 1. Load v0 baseline and compute starting scores
with open(BASELINE_OUTPUT_PATH, "r", encoding="utf-8") as f:
    v0_data = json.load(f)

v0_scores = {"factual": 0, "reasoning": 0, "identity": 0}
category_counts = {"factual": 20, "reasoning": 20, "identity": 10}

for item in v0_data:
    passed_v0 = any(kw.lower() in item.get("v0_response", "").lower() for kw in item["expected_keywords"])
    if passed_v0:
        v0_scores[item["category"]] += 1

v0_pct = {cat: (v0_scores[cat] / category_counts[cat]) * 100 for cat in category_counts}
print(f"Baseline (v0) -> Factual: {v0_pct['factual']}%, Reasoning: {v0_pct['reasoning']}%, Identity: {v0_pct['identity']}%")

# 2. Evaluate Candidate
print("\nEvaluating candidate model against Anchor Benchmark...")
candidate_scores = {"factual": 0, "reasoning": 0, "identity": 0}

for item in tqdm(benchmark_data, desc="Candidate Inference"):
    messages =[
        {"role": "system", "content": "You are a helpful, harmless, and honest AI assistant."},
        {"role": "user", "content": item["question"]}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=256, 
        use_cache=True, 
        do_sample=False,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).lower()
    
    passed = any(kw.lower() in response for kw in item["expected_keywords"])
    if passed:
        candidate_scores[item["category"]] += 1

candidate_pct = {cat: (candidate_scores[cat] / category_counts[cat]) * 100 for cat in category_counts}

print("\n--- Candidate Scores ---")
print(f"Factual:  {candidate_pct['factual']:.1f}%")
print(f"Reasoning:{candidate_pct['reasoning']:.1f}%")
print(f"Identity: {candidate_pct['identity']:.1f}%")

# 3. MVL Binary Rollback Logic
identity_drop = v0_pct['identity'] - candidate_pct['identity']
reasoning_drop = v0_pct['reasoning'] - candidate_pct['reasoning']

system_state = "ACCEPT"
rollback_stage = 0  # 0 means accepted, 1 means reverted (binary for MVL)

if identity_drop > 1.0 or reasoning_drop > 5.0:
    system_state = "REJECT"
    rollback_stage = 1
    print("\n❌ GATE FAILED: Catastrophic forgetting detected.")
    if identity_drop > 1.0: print(f"   ↳ Identity dropped by {identity_drop:.1f}% (> 1%)")
    if reasoning_drop > 5.0: print(f"   ↳ Reasoning dropped by {reasoning_drop:.1f}% (> 5%)")
    print("   ↳ ACTION: Adapter v1 discarded. Reverting to v0.")
else:
    print("\n✅ GATE PASSED: Model retained core knowledge safely.")

In [ ]:
# ==============================================================================
# BLOCK 07 (MVL COMPLIANT) — Merge & 5-Metric Logging
# ==============================================================================
import hashlib
import subprocess
import datetime
import os
import sys
import jsonlines

print("=" * 60)
print("  BLOCK 07: MERGE & MVL LOGGING")
print("=" * 60)

MERGED_DIR = "merged_model_v1"

if system_state == "ACCEPT":
    print("Executing DARE-TIES Merge (seed=42, density=0.70)...")
    
    # Free up memory before mergekit
    del model
    del trainer
    import torch
    torch.cuda.empty_cache()
    
    merge_cmd_python =[sys.executable, "-m", "mergekit.scripts.merge", "merge_spec.yaml", MERGED_DIR, "--copy-tokenizer", "--cuda", "--low-cpu-memory"]
    merge_cmd_cli =["mergekit-merge", "merge_spec.yaml", MERGED_DIR, "--copy-tokenizer", "--cuda", "--low-cpu-memory"]
    
    try:
        subprocess.run(merge_cmd_python, check=True)
        print(f"✅ Merge successful! New base model saved to ./{MERGED_DIR}")
    except subprocess.CalledProcessError:
        print("⚠️ Python module invocation failed. Falling back to CLI 'mergekit-merge'...")
        try:
            subprocess.run(merge_cmd_cli, check=True)
            print(f"✅ Merge successful! New base model saved to ./{MERGED_DIR}")
        except subprocess.CalledProcessError:
            print("❌ Merge failed entirely. Please check mergekit installation.")
            sys.exit(1)
else:
    print("Skipping merge due to Rollback.")

# Generate SHA of the adapter
adapter_file_path = os.path.join(ADAPTER_SAVE_PATH, "adapter_model.safetensors")
with open(adapter_file_path, "rb") as f:
    adapter_sha = hashlib.sha256(f.read()).hexdigest()

# Calculate overall Fast Score Delta (Average of the 3 categories for Cycle 1 simplicity)
overall_v0_score = sum(v0_pct.values()) / 3
overall_candidate_score = sum(candidate_pct.values()) / 3
fast_score_delta = overall_candidate_score - overall_v0_score

# 5 Metrics STRICTLY Required for Cycle 1 MVL (Section 23.1)
log_entry = {
    # Metadata
    "version": "v1",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "adapter_sha256": adapter_sha,
    "system_state": system_state,
    
    # MVL 5 Metrics
    "replay_ratio": 0.70,                  # Hardcoded for Cycle 1
    "fast_score_delta": fast_score_delta,  # Consolidated into one metric
    "rollback_stage": rollback_stage,      # 0 or 1
    "kl_from_anchor": 0.0,                 # Placeholder, valid for MVL
    "domain_gain": candidate_pct['factual'] - v0_pct['factual'] # Using factual as proxy for domain gain in Cycle 1
}

with jsonlines.open("update_log.jsonl", mode="a") as writer:
    writer.write(log_entry)

print("\n✅ Cycle 1 logged successfully to update_log.jsonl")
print(f"🔒 Adapter Hash: {adapter_sha}")
print("🎉 MINIMUM VIABLE LOOP (MVL) FULLY SECURED AND COMPLETE! 🎉")